
# Predicción espacio-temporal adaptativa de casos de malaria en el Perú

**Notebook 01 — Carga, limpieza, agregación y línea base (baseline)**

Este notebook cubre la primera fase del proyecto:

1. Carga del dataset de vigilancia (línea de casos individuales, MINSA 2009–2024).
2. Limpieza de inconsistencias identificadas en la exploración inicial (edades corruptas, duplicados, unidades de edad mixtas).
3. Definición del universo espacial de estudio (distritos endémicos).
4. Agregación a la unidad de análisis del proyecto: **UBIGEO × semana epidemiológica**.
5. Completado de la grilla espacio-temporal (semanas sin casos reportados deben quedar en 0, no ausentes).
6. Exploración descriptiva mínima (proporción de ceros, series por departamento).
7. Función de partición temporal *walk-forward* (rolling origin) para validación, respetando el orden temporal.
8. Modelo línea base: GLM Binomial Negativo (single-predictor / autoregresivo simple), como punto de comparación para los modelos posteriores (ML, ST-GNN, híbrido/adaptativo).

> **Nota:** este notebook es la línea base. Los notebooks siguientes (02_modelos_ml.ipynb, 03_stgnn.ipynb, 04_modelo_adaptativo.ipynb) reutilizarán el dataset agregado que se guarda al final de este notebook.


## 1. Configuración e importación de librerías

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 4)

# --- Rutas (ajustar según tu entorno local) ---
DATA_PATH = "datos_abiertos_vigilancia_malaria_2009_2024.csv"
OUTPUT_DIR = "outputs"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 2. Carga del dataset (línea de casos)

In [ ]:

df = pd.read_csv(DATA_PATH, encoding="utf-8", dtype=str)

print("Shape:", df.shape)
df.head()


In [ ]:

# Tipos correctos para columnas numéricas clave
df["ano"] = df["ano"].astype(int)
df["semana"] = df["semana"].astype(int)

# Semana 53 solo existe en años con 53 semanas ISO; se mantiene tal cual por ahora,
# se decide más adelante si se colapsa con la semana 52 según el calendario epidemiológico del Perú (MINSA).
print(df["ano"].min(), df["ano"].max())
print(sorted(df["semana"].unique()))



## 3. Limpieza de datos

Basado en la exploración inicial, se identificaron 3 problemas a resolver:

1. **`edad` con valores imposibles** (hasta ~90 millones), probablemente dígitos de DNI insertados por error.
2. **`tipo_edad` mixto** (A = años, M = meses, D = días) — se homogeniza a años decimales.
3. **Filas duplicadas exactas** (~30,846) — no hay ID de caso único, así que se documentan y se deja una bandera en vez de eliminarlas silenciosamente.


In [ ]:

df["edad_num"] = pd.to_numeric(df["edad"], errors="coerce")

# Umbral de edad plausible (0-120 años). Se homogeniza tipo_edad a años.
def edad_a_anios(row):
    e, t = row["edad_num"], row["tipo_edad"]
    if pd.isna(e):
        return np.nan
    if t == "A":
        return e
    if t == "M":
        return e / 12
    if t == "D":
        return e / 365
    return np.nan

df["edad_anios"] = df.apply(edad_a_anios, axis=1)

n_invalidas = (df["edad_anios"] > 120).sum() | (df["edad_anios"] < 0).sum()
print("Registros con edad implausible (>120 años):", (df["edad_anios"] > 120).sum())

# Se marcan como NaN las edades no plausibles, sin eliminar la fila
# (para este proyecto, la edad NO es la variable objetivo; se conserva el caso, solo se invalida la edad).
df.loc[df["edad_anios"] > 120, "edad_anios"] = np.nan


In [ ]:

# Duplicados exactos: se documentan, no se eliminan automáticamente sin criterio adicional.
dup_mask = df.duplicated(keep=False)
print(f"Filas involucradas en duplicados exactos: {dup_mask.sum()} ({dup_mask.mean()*100:.2f}% del total)")

# Bandera para trazabilidad (decisión metodológica documentada en el protocolo, sección de limitaciones)
df["es_duplicado_exacto"] = df.duplicated(keep=False)

# NOTA METODOLÓGICA: para el conteo agregado por UBIGEO-semana, cada fila representa
# un caso reportado. Si se decide que los duplicados exactos son errores de captura
# (no personas distintas), se debe usar `keep="first"` antes de agregar.
# Por defecto, en este notebook SÍ se cuentan (asume que son casos distintos),
# pero queda como parámetro explícito para el análisis de sensibilidad.
DEDUPLICAR = False

if DEDUPLICAR:
    df = df.drop_duplicates(keep="first")
    print("Nuevo shape tras deduplicar:", df.shape)



## 4. Definición del universo espacial de estudio

De los ~1,874 distritos del Perú, solo **396 UBIGEOs** reportaron al menos un caso en 2009–2024,
y **Loreto concentra ~86.6%** del total nacional. Se define el universo de estudio como los distritos
con historial de transmisión, evitando incluir miles de distritos en cero perpetuo que no aportan señal
y solo inflan artificialmente el problema de exceso de ceros.

Criterio de inclusión (ajustable): distrito con al menos **N casos acumulados** en todo el periodo.


In [ ]:

casos_por_ubigeo = df.groupby("ubigeo").size().sort_values(ascending=False)

print("Distribución de casos acumulados por distrito (top 15):")
print(casos_por_ubigeo.head(15))

# Umbral de inclusión: se propone un mínimo de casos acumulados en 16 años.
# Ajustar UMBRAL_MIN_CASOS según el análisis de sensibilidad que se quiera reportar.
UMBRAL_MIN_CASOS = 20

ubigeos_endemicos = casos_por_ubigeo[casos_por_ubigeo >= UMBRAL_MIN_CASOS].index.tolist()
print(f"\nDistritos incluidos con umbral >= {UMBRAL_MIN_CASOS} casos: {len(ubigeos_endemicos)} de {df['ubigeo'].nunique()}")

df_estudio = df[df["ubigeo"].isin(ubigeos_endemicos)].copy()
print("Cobertura de casos retenida:", f"{len(df_estudio) / len(df) * 100:.2f}%")



## 5. Agregación a UBIGEO × semana epidemiológica

Se generan conteos semanales por distrito, separando por especie
(`MALARIA POR P. VIVAX` / `MALARIA P. FALCIPARUM`) y también el total combinado.


In [ ]:

agg = (
    df_estudio
    .groupby(["ubigeo", "departamento", "provincia", "distrito", "ano", "semana", "enfermedad"])
    .size()
    .reset_index(name="casos")
)

# Pivot para tener P. vivax y P. falciparum como columnas, más el total
agg_wide = (
    agg
    .pivot_table(
        index=["ubigeo", "departamento", "provincia", "distrito", "ano", "semana"],
        columns="enfermedad",
        values="casos",
        fill_value=0,
    )
    .reset_index()
)

agg_wide.columns.name = None
agg_wide = agg_wide.rename(columns={
    "MALARIA POR P. VIVAX": "casos_vivax",
    "MALARIA P. FALCIPARUM": "casos_falciparum",
})
agg_wide["casos_total"] = agg_wide.get("casos_vivax", 0) + agg_wide.get("casos_falciparum", 0)

agg_wide.head()



## 6. Completar la grilla espacio-temporal (semanas sin casos = 0)

`agg_wide` solo contiene combinaciones UBIGEO-semana **con al menos un caso**.
Para modelar correctamente la naturaleza de conteo (incluyendo el exceso de ceros),
es indispensable expandir a **todas** las semanas epidemiológicas del periodo para
cada distrito del universo de estudio, y rellenar con 0 donde no hubo reporte.


In [ ]:

# Construcción de la grilla completa: todos los UBIGEOs endémicos x todos los (año, semana) observados
anios_semanas = (
    df_estudio[["ano", "semana"]]
    .drop_duplicates()
    .sort_values(["ano", "semana"])
    .reset_index(drop=True)
)

ubigeo_meta = (
    df_estudio[["ubigeo", "departamento", "provincia", "distrito"]]
    .drop_duplicates(subset=["ubigeo"])
)

# Producto cartesiano UBIGEO x (año, semana)
grid = ubigeo_meta.merge(anios_semanas, how="cross")

panel = grid.merge(
    agg_wide.drop(columns=["departamento", "provincia", "distrito"]),
    on=["ubigeo", "ano", "semana"],
    how="left",
)

for col in ["casos_vivax", "casos_falciparum", "casos_total"]:
    if col not in panel.columns:
        panel[col] = 0
    panel[col] = panel[col].fillna(0).astype(int)

panel = panel.sort_values(["ubigeo", "ano", "semana"]).reset_index(drop=True)

print("Shape del panel completo:", panel.shape)
panel.head()


In [ ]:

# Chequeo de exceso de ceros (justifica el uso de modelos de conteo / zero-inflated)
prop_ceros = (panel["casos_total"] == 0).mean()
print(f"Proporción de observaciones UBIGEO-semana en cero: {prop_ceros*100:.2f}%")

print("\nEstadísticos descriptivos de casos_total:")
print(panel["casos_total"].describe())

# Varianza vs media (sobredispersión): si var >> media, se confirma la necesidad de
# binomial negativa en vez de Poisson.
media = panel["casos_total"].mean()
varianza = panel["casos_total"].var()
print(f"\nMedia: {media:.3f} | Varianza: {varianza:.3f} | Ratio Var/Media: {varianza/media:.2f}")


## 7. Exploración descriptiva mínima

In [ ]:

serie_nacional = panel.groupby(["ano", "semana"])["casos_total"].sum().reset_index()
serie_nacional["t"] = range(len(serie_nacional))

plt.plot(serie_nacional["t"], serie_nacional["casos_total"])
plt.title("Casos totales agregados (universo endémico) por semana epidemiológica")
plt.xlabel("Semana (índice continuo 2009-2024)")
plt.ylabel("N° de casos")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/serie_nacional_semanal.png", dpi=150)
plt.show()


In [ ]:

top_distritos = (
    panel.groupby("distrito")["casos_total"].sum().sort_values(ascending=False).head(8).index
)

fig, ax = plt.subplots(figsize=(11, 5))
for d in top_distritos:
    serie = panel[panel["distrito"] == d].groupby(["ano", "semana"])["casos_total"].sum().reset_index()
    serie["t"] = range(len(serie))
    ax.plot(serie["t"], serie["casos_total"], label=d, alpha=0.8)

ax.set_title("Series semanales de casos — top 8 distritos con más casos acumulados")
ax.set_xlabel("Semana (índice continuo)")
ax.set_ylabel("N° de casos")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/series_top_distritos.png", dpi=150)
plt.show()



## 8. Partición temporal walk-forward (rolling origin)

Para evaluar predicción **multihorizonte** sin fuga de información (*data leakage*),
la validación debe respetar el orden temporal: el modelo solo puede entrenarse con
datos anteriores al punto de corte y probarse en semanas futuras.

La función siguiente genera una lista de splits `(train_idx, test_idx)` deslizando el
punto de corte hacia adelante, para múltiples orígenes de predicción.


In [ ]:

def generar_splits_walk_forward(fechas_ordenadas, horizonte_semanas=4, min_train_semanas=104, paso=4):
    '''
    fechas_ordenadas: array-like de tuplas (ano, semana) ordenadas cronológicamente y únicas.
    horizonte_semanas: cuántas semanas hacia adelante se evalúa la predicción (multihorizonte).
    min_train_semanas: tamaño mínimo de la ventana de entrenamiento (ej. 104 = 2 años).
    paso: cada cuántas semanas se desliza el origen de predicción.

    Retorna una lista de dicts con train_end_idx y test_idx (posiciones sobre fechas_ordenadas).
    '''
    splits = []
    n = len(fechas_ordenadas)
    origen = min_train_semanas
    while origen + horizonte_semanas <= n:
        splits.append({
            "train_end_idx": origen - 1,          # última semana de entrenamiento (inclusive)
            "test_idx": origen + horizonte_semanas - 1,  # semana objetivo a horizonte fijo
        })
        origen += paso
    return splits


fechas_unicas = (
    panel[["ano", "semana"]]
    .drop_duplicates()
    .sort_values(["ano", "semana"])
    .reset_index(drop=True)
)
fechas_unicas["idx_tiempo"] = range(len(fechas_unicas))

splits = generar_splits_walk_forward(fechas_unicas, horizonte_semanas=4, min_train_semanas=104, paso=4)
print(f"Número de splits walk-forward generados (horizonte=4 semanas): {len(splits)}")
print("Ejemplo de los primeros 3 splits:", splits[:3])



## 9. Modelo línea base: GLM Binomial Negativo

Se construye un primer modelo simple —autoregresivo de 1 predictor (casos de la semana t-h)—
únicamente como **línea base de referencia**, análogo en espíritu al GLM de un solo predictor
usado en el paper de referencia (Pan et al., 2026), pero aquí a nivel distrital.

Este modelo NO es el modelo final del proyecto — es el punto de comparación mínimo que
los modelos posteriores (ML, ST-GNN, adaptativo) deben superar.


In [ ]:

# pip install statsmodels si no está disponible
import statsmodels.api as sm
import statsmodels.formula.api as smf

HORIZONTE = 4  # semanas de anticipación

# Ejemplo con un solo distrito para validar el pipeline antes de escalar a todos
distrito_ejemplo = top_distritos[0]
serie_d = (
    panel[panel["distrito"] == distrito_ejemplo]
    .groupby(["ano", "semana"])["casos_total"].sum()
    .reset_index()
    .sort_values(["ano", "semana"])
    .reset_index(drop=True)
)

serie_d["casos_lag"] = serie_d["casos_total"].shift(HORIZONTE)
serie_d = serie_d.dropna().reset_index(drop=True)

n = len(serie_d)
corte = int(n * 0.8)
train, test = serie_d.iloc[:corte], serie_d.iloc[corte:]

modelo = smf.glm(
    formula="casos_total ~ casos_lag",
    data=train,
    family=sm.families.NegativeBinomial(),
).fit()

print(modelo.summary())

pred = modelo.predict(test)
rmse = np.sqrt(np.mean((test["casos_total"].values - pred.values) ** 2))
corr = np.corrcoef(test["casos_total"].values, pred.values)[0, 1]

print(f"\nDistrito de ejemplo: {distrito_ejemplo}")
print(f"RMSE (horizonte {HORIZONTE} semanas): {rmse:.2f}")
print(f"Correlación (horizonte {HORIZONTE} semanas): {corr:.3f}")



## 10. Guardar el panel agregado para los siguientes notebooks

El resto de modelos (Random Forest / ML clásico, LSTM/temporal, ST-GNN, modelo adaptativo)
deben partir del **mismo panel limpio y agregado**, para que la comparación entre modelos
sea válida (mismos datos, mismos splits).


In [ ]:

panel.to_parquet(f"{OUTPUT_DIR}/panel_ubigeo_semana.parquet", index=False)
panel.to_csv(f"{OUTPUT_DIR}/panel_ubigeo_semana.csv", index=False)

print("Panel guardado en:", OUTPUT_DIR)
print("Shape final:", panel.shape)
print("Columnas:", list(panel.columns))



## Próximos pasos (siguientes notebooks)

1. **`02_modelos_conteo_y_ml.ipynb`**: comparar GLM Binomial Negativo (todos los distritos, no solo el ejemplo), modelos zero-inflated/hurdle, y Random Forest / Gradient Boosting como baseline de ML, todos evaluados con los mismos splits walk-forward multihorizonte.
2. **`03_modelos_temporales.ipynb`**: LSTM / modelos secuenciales por distrito o pooled.
3. **`04_stgnn.ipynb`**: construcción del grafo espacial (probar 2–3 definiciones: contigüidad, distancia, proxy de movilidad) + arquitectura ST-GNN.
4. **`05_modelo_adaptativo.ipynb`**: mecanismo de detección/adaptación a cambios de patrón epidemiológico (drift) integrado al mejor modelo espacio-temporal.
5. **`06_comparacion_final.ipynb`**: tabla comparativa de todos los modelos por horizonte de predicción (RMSE, correlación, y alguna métrica de calibración de intervalos si se agregan modelos probabilísticos).

**Decisiones que quedaron abiertas y deben resolverse antes de escalar este notebook:**
- Umbral `UMBRAL_MIN_CASOS` para definir el universo endémico (actualmente 20; hacer análisis de sensibilidad).
- `DEDUPLICAR` = True/False para las 30,846 filas duplicadas exactas.
- Modelar `casos_vivax`, `casos_falciparum` o `casos_total` (o los tres, comparando).
- Tratamiento de la semana 53 (colapsar con semana 52 según calendario epidemiológico MINSA, o dejarla aparte).
